In [1]:
!pip install gliner

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 10.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 10.1 MB/s  0:00:002.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 8.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 6.7 MB/s  0:01:54 eta 0:00:010:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 9.8 MB/s  0:00:01s eta 0:00:010:0101
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 8.5 MB/s  0:01:06 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 9.5 MB/s  0:00:01m 10.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.9

In [4]:
!pip install pandas

  Using cached pandas-3.0.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)


In [5]:
# Core
import torch
import pandas as pd
import numpy as np

# NER
from gliner import GLiNER

# Utils
from typing import List, Dict


/home/kevin-shah/Desktop/trading/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]━━ 1/2 [openpyxl]


In [11]:
# Load tweets from Excel (use local path)
df = pd.read_excel("/home/kevin-shah/Desktop/trading/trading/trump_tweets_sota_classified(1).xlsx")

# Adjust column name if needed
texts = df["tweet_text"].dropna().tolist()

print(f"Loaded {len(texts)} tweets")
print(f"Sample tweet: {texts[0][:200] if texts else 'No tweets found'}")

Loaded 255 tweets
Sample tweet: washingtonexaminer.com/restori


In [ ]:
# Load a better GLiNER model (large-v2.1 has much better entity recognition)
model = GLiNER.from_pretrained("urchade/gliner_large-v2.1")

# Use GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"GLiNER large-v2.1 loaded on {device}")

/home/kevin-shah/Desktop/trading/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [01:21<00:00, 20.39s/it]


GLiNER loaded on cpu


In [13]:
NER_LABELS = [
    "PERSON",
    "ORG",
    "COUNTRY",
    "LOCATION",
    "COMMODITY",
    "CURRENCY",
    "SECTOR",
    "EVENT"
]


In [ ]:
import re

def clean_tweet(text: str) -> str:
    """Clean tweet text for better NER performance."""
    if not isinstance(text, str):
        return ""
    # Remove URLs (including bare domains like example.com/path)
    text = re.sub(r'https?://\S+', '', text)  # http:// or https://
    text = re.sub(r'www\.\S+', '', text)       # www.
    text = re.sub(r'\S+\.(com|org|net|gov|io|co|edu|info|biz|me)/\S*', '', text)  # bare domains with path
    text = re.sub(r'\S+\.(com|org|net|gov|io|co|edu|info|biz|me)\b', '', text)    # bare domains without path
    # Remove mentions but keep the name
    text = re.sub(r'@(\w+)', r'\1', text)
    # Remove hashtag symbol but keep the word
    text = re.sub(r'#(\w+)', r'\1', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_valid_tweet(text: str) -> bool:
    """Check if tweet has meaningful text content (not just a URL)."""
    if not isinstance(text, str):
        return False
    cleaned = clean_tweet(text)
    # Must have at least 10 chars of actual content after cleaning
    return len(cleaned) >= 10

def extract_entities(
    text: str,
    labels: List[str] = NER_LABELS,
    threshold: float = 0.4  # Can use higher threshold with better model
) -> List[Dict]:
    """
    Run NER on a single tweet_text.
    Returns list of entity dictionaries.
    """
    if not isinstance(text, str) or not text.strip():
        return []
    
    # Clean the tweet first
    cleaned_text = clean_tweet(text)
    
    # Skip if no meaningful content after cleaning
    if len(cleaned_text) < 10:
        return []

    # Use flat_ner=True for better performance on short texts like tweets
    preds = model.predict_entities(
        cleaned_text,
        labels=labels,
        threshold=threshold,
        flat_ner=True  # Better for social media text
    )

    return [
        {
            "entity": p["text"],
            "label": p["label"],
            "score": round(p["score"], 3),
            "start": p["start"],
            "end": p["end"]
        }
        for p in preds
    ]

In [18]:
# Check how many tweets are just URLs vs actual text
url_only_count = sum(1 for t in df["tweet_text"] if not is_valid_tweet(t))
valid_count = sum(1 for t in df["tweet_text"] if is_valid_tweet(t))

print(f"Total tweets: {len(df)}")
print(f"URL-only tweets (skipped): {url_only_count}")
print(f"Valid text tweets: {valid_count}")
print(f"\nExamples of URL-only tweets:")
for i, t in enumerate(df["tweet_text"]):
    if not is_valid_tweet(t):
        print(f"  - {str(t)[:60]}")
        if i > 4:
            break

Total tweets: 255
URL-only tweets (skipped): 50
Valid text tweets: 205

Examples of URL-only tweets:
  - washingtonexaminer.com/restori
  - breitbart.com/politics/2025/06
  - redstate.com/redstate-guest-ed
  - foxnews.com/opinion/loeffler-t
  - nypost.com/2025/06/28/us-news/
  - whitehouse.gov/articles/2025/0


In [19]:
# Find a good sample tweet with actual text content
print("Examples of valid tweets with text content:\n")
for i, row in df.iterrows():
    text = row["tweet_text"]
    if is_valid_tweet(text):
        cleaned = clean_tweet(text)
        print(f"Tweet {i}:")
        print(f"  Original: {text[:100]}...")
        print(f"  Cleaned:  {cleaned[:100]}...")
        entities = extract_entities(text)
        if entities:
            print(f"  Entities:")
            for e in entities:
                print(f"    → {e['label']}: {e['entity']} ({e['score']})")
        else:
            print("  Entities: None found")
        print()
        if i > 15:  # Show first few valid examples
            break

Examples of valid tweets with text content:

Tweet 5:
  Original: I apologize for the long wait on the Faith Leaders Conference Call. AT&T ought to get its act togeth...
  Cleaned:  I apologize for the long wait on the Faith Leaders Conference Call. AT&T ought to get its act togeth...
  Entities: None found

Tweet 6:
  Original: I’m doing a major Conference Call with Faith Leaders from all over the Country, and AT&T is totally ...
  Cleaned:  I’m doing a major Conference Call with Faith Leaders from all over the Country, and AT&T is totally ...
  Entities: None found

Tweet 7:
  Original: To show people how spoiled Countries have become with respect to the United States of America, and I...
  Cleaned:  To show people how spoiled Countries have become with respect to the United States of America, and I...
  Entities:
    → CURRENCY: RICE (0.264)

Tweet 9:
  Original: Jerome “Too Late” Powell, and his entire Board, should be ashamed of themselves for allowing this to...
  Cleaned:  Jerom

In [20]:
df["entities"] = df["tweet_text"].apply(extract_entities)


In [ ]:
df[["tweet_id", "tweet_text", "entities"]].head(5)


,tweet_id,tweet_text,entities
0,f6b6bc55c8b5,washingtonexaminer.com/restori,"[{'entity': 'washingtonexaminer.com/restori', ..."
1,362a7010b3c1,breitbart.com/politics/2025/06,[]
2,65c19d0b7fb5,redstate.com/redstate-guest-ed,[]
3,8f0d572a979f,foxnews.com/opinion/loeffler-t,"[{'entity': 'foxnews.com', 'label': 'ORG', 'sc..."
4,707c1b8d12dd,nypost.com/2025/06/28/us-news/,"[{'entity': 'nypost.com', 'label': 'ORG', 'sco..."


In [ ]:
flat_entities = []

for _, row in df.iterrows():
    for ent in row["entities"]:
        flat_entities.append({
            "tweet_id": row["tweet_id"],
            "entity": ent["entity"].lower(),
            "label": ent["label"],
            "score": ent["score"]
        })

ner_df = pd.DataFrame(flat_entities)
ner_df.head(10)
ner_df


,tweet_id,entity,label,score
0,f6b6bc55c8b5,washingtonexaminer.com/restori,ORG,0.534
1,8f0d572a979f,foxnews.com,ORG,0.479
2,8f0d572a979f,loeffler-t,PERSON,0.513
3,707c1b8d12dd,nypost.com,ORG,0.875
4,9e27945fb8cb,faith leaders conference call,EVENT,0.980
...,...,...,...,...
826,68c3cb276bab,idaho,LOCATION,0.990
827,68c3cb276bab,state,LOCATION,0.776
828,68c3cb276bab,2016,EVENT,0.565
829,68c3cb276bab,2020,EVENT,0.636


In [ ]:
ner_df.groupby(["label", "entity"]) \
      .size() \
      .sort_values(ascending=False) \
      .head(30)


label     entity         
COUNTRY   iran               44
          america            18
ORG       truthsocial.com    13
COUNTRY   israel             12
          united states      11
LOCATION  los angeles        10
ORG       youtube             8
COUNTRY   china               8
          u.s.                8
          country             7
PERSON    trump               6
LOCATION  nuclear sites       6
EVENT     peace               6
ORG       national guard      5
          military            5
EVENT     deal                5
LOCATION  country             5
PERSON    president trump     5
          biden               4
          donald trump        4
ORG       fbi                 4
          republicans         4
          nato                4
          maga                4
CURRENCY  bill                4
ORG       army                4
          cnn                 4
          congress            4
PERSON    djt                 4
          donald j. trump     4
dtype: int64